# Evaluación Formativa 2. Práctica de modelamiento predictivo
## Framingham Heart Study: riesgo de enfermedad coronaria a 10 años

MCDI501 · Grupo 3 · Integrantes: Felipe Díaz Toro, Pablo Ríos Passteni, Guillermo Subiabre Chacón, Ninoska Yévenes Hernández


## Propósito de esta formativa

Esta formativa es la **práctica previa a la Sumativa 3**: ensayamos el flujo completo de modelamiento predictivo con un modelo simple, para recibir retroalimentación antes del proyecto final. Es una versión simplificada de S3: **un** modelo logístico, imputación simple de faltantes y sin bootstrap de coeficientes.

### El problema

La enfermedad coronaria (CHD) es una de las principales causas de enfermedad cardiovascular. Con el **Framingham Heart Study** (4.238 participantes, 16 variables) queremos **anticipar quién desarrollará CHD en los 10 años siguientes** (`TenYearCHD`: 1 = sí, 0 = no), a partir de variables medidas al inicio del seguimiento (edad, sexo, presión arterial, glucosa, tabaquismo, etc.). Solo el 15,2 % de los participantes desarrolla CHD, así que las clases están **desbalanceadas**, y eso condiciona cómo se interpretan las métricas.

## 0. Configuración y datos

La primera celda importa las librerías, **ubica el repositorio y carga el dataset** `data/raw/framingham.csv`. Antes de usarlo verifica su huella SHA-256 y su forma (4.238 × 16), para asegurar que es exactamente la misma copia de S1 y S2 y que los resultados sean comparables. La segunda celda fija la semilla (`SEED = 42`, para reproducibilidad), define la variable objetivo y el estilo de las figuras, e imprime la prevalencia de CHD.

In [3]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, auc)

# Ubicar el repositorio sin depender de rutas relativas
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "requirements.txt").exists() and (p / "redaccion").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Abra el notebook dentro del repositorio del proyecto.")

# Copia versionada del dataset: se verifica para que los resultados sean los de S1 y S2
CSV = ROOT / "data/raw/framingham.csv"
SHA_ESPERADO = "40921748dbdf6587045b9822414199a1ae3349358b63b975ab2208aa3cccc45b"
FORMA_ESPERADA = (4238, 16)
sha = hashlib.sha256(CSV.read_bytes()).hexdigest()
df = pd.read_csv(CSV)
if sha != SHA_ESPERADO or df.shape != FORMA_ESPERADA:
    raise ValueError("data/raw/framingham.csv no es la copia del proyecto. "
                     "Restáurela con: git checkout -- data/raw/framingham.csv")
print(f"Dataset verificado: {df.shape[0]:,} filas x {df.shape[1]} columnas".replace(",", "."))
print("SHA-256:", sha)

SEED = 42
np.random.seed(SEED)
TARGET = "TenYearCHD"

# Misma paleta y estilo que S1 y S2
NAVY, RED, GREY, BG, OCHRE = "#0d2741", "#ae0000", "#8c8578", "#f6f1ea", "#c0852f"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.facecolor": BG,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": "#ddd5c6", "axes.axisbelow": True,
})

print(f"Prevalencia de {TARGET}: {df[TARGET].mean():.1%} ({int(df[TARGET].sum())} de {len(df):,} casos)".replace(",", "."))

Dataset verificado: 4.238 filas x 16 columnas
SHA-256: 40921748dbdf6587045b9822414199a1ae3349358b63b975ab2208aa3cccc45b
Prevalencia de TenYearCHD: 15.2% (644 de 4.238 casos)


## 1. Preparación de datos para modelamiento (21 pts)
### 1.1 Hallazgos de S1 y S2
El punto de partida no es el dato crudo, sino lo aprendido en S1 (exploración e inferencia) y S2 (validación con remuestreo). La tabla compara la correlación de cada variable candidata con `TenYearCHD` (S1) y, cuando existe, su IC bootstrap al 95 % de S2.

In [4]:
# Correlaciones de Pearson con TenYearCHD (casos disponibles de cada par), como en S1
candidatas = ["age", "male", "sysBP", "cigsPerDay", "glucose", "totChol", "BMI", "prevalentHyp", "diabetes"]
corr_s1 = df[candidatas + [TARGET]].corr()[TARGET].drop(TARGET)

# Validación bootstrap de S2 (IC percentil 95 %, 10.000 remuestras); solo se validaron estas
S2_IC = {"age": [0.197, 0.254], "sysBP": [0.183, 0.250]}

tabla = pd.DataFrame({
    "r con CHD (S1)": corr_s1.round(3),
    "IC bootstrap (S2)": [str(S2_IC[v]) if v in S2_IC else "no validada en S2" for v in corr_s1.index],
}).sort_values("r con CHD (S1)", ascending=False)
tabla

,r con CHD (S1),IC bootstrap (S2)
age,0.225,"[0.197, 0.254]"
sysBP,0.216,"[0.183, 0.25]"
prevalentHyp,0.178,no validada en S2
glucose,0.126,no validada en S2
diabetes,0.097,no validada en S2
male,0.088,no validada en S2
totChol,0.082,no validada en S2
BMI,0.075,no validada en S2
cigsPerDay,0.058,no validada en S2


**Lectura.** Las variables más asociadas con CHD son la edad (r = 0,225) y la presión sistólica (r = 0,216), y S2 confirmó que ambas son estables: sus IC bootstrap [0,197; 0,254] y [0,183; 0,250] son estrechos y no incluyen el cero. Les siguen hipertensión prevalente (0,178), glucosa (0,126), diabetes (0,097) y sexo masculino (0,088); cigarrillos por día (0,058) es la más baja. Todas las correlaciones son modestas (la máxima es 0,23), lo esperable en un fenómeno multifactorial a 10 años: ninguna variable predice bien por sí sola, por eso se modelan en conjunto. Además, S2 solo validó con bootstrap cinco correlaciones; las demás candidatas no pasaron por esa validación y se declara explícitamente en la tabla. Para el tabaquismo, S2 mostró que fumador actual–CHD (r = 0,019) no se distingue de cero.

**Faltantes y valores extremos (S1).** Se revisan los datos faltantes y los valores extremos que S1 ya había caracterizado, porque condicionan el tratamiento de la sección 1.2.

In [5]:
falt = df.isna().sum()
falt = pd.DataFrame({"faltantes": falt, "%": (100 * falt / len(df)).round(1)}).query("faltantes > 0")
display(falt)

# Valores extremos por la regla de Tukey en las variables que usaremos
def pct_tukey(x):
    x = x.dropna(); q1, q3 = x.quantile([.25, .75]); i = q3 - q1
    return 100 * ((x < q1 - 1.5 * i) | (x > q3 + 1.5 * i)).mean()
print({v: round(pct_tukey(df[v]), 1) for v in ["sysBP", "glucose", "cigsPerDay", "age"]})

,faltantes,%
education,105,2.5
cigsPerDay,29,0.7
BPMeds,53,1.3
totChol,50,1.2
BMI,19,0.4
heartRate,1,0.0
glucose,388,9.2


{'sysBP': np.float64(3.0), 'glucose': np.float64(4.9), 'cigsPerDay': np.float64(0.3), 'age': np.float64(0.0)}


**Lectura.** Hay faltantes en siete variables, ninguna sobre el 10 %: la glucosa concentra la mayor parte (388 casos, 9,2 %), seguida de educación (105, 2,5 %); los 29 de cigarrillos por día (0,7 %) son todos fumadores actuales (S1). Entre los predictores que usaremos, solo glucosa y cigarrillos por día tienen faltantes. La regla de Tukey marca 3,0 % de valores extremos en la presión sistólica, 4,9 % en la glucosa, 0,3 % en cigarrillos por día y 0 % en la edad. S1 mostró que son plausibles (los registros de glucosa de 200 mg/dL o más corresponden a personas diabéticas), por eso no se eliminan: se acota la cola de la glucosa en la sección 1.2, como recomendó S2.

### 1.2 Tratamiento de faltantes, partición y estandarización
Estrategia simple, en este orden: (1) partición 70/30 estratificada por `TenYearCHD`, (2) tope de la glucosa, (3) imputación con medianas y (4) estandarización. Todo se calcula con el conjunto de entrenamiento y se aplica al de prueba.

In [6]:
predictores = ["age", "sysBP", "male", "cigsPerDay", "glucose"]
X = df[predictores].copy()
y = df[TARGET]

# 1) Partición 70/30 estratificada ANTES de imputar o estandarizar (evita fuga de información al test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}".replace(",", "."))
print("Prevalencia train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))

# 2) Glucosa: acotar al P99 calculado SOLO en train (S2 recomendó ~175 mg/dL)
tope = X_train["glucose"].quantile(0.99)
X_train["glucose"] = X_train["glucose"].clip(upper=tope)
X_test["glucose"] = X_test["glucose"].clip(upper=tope)
print(f"Tope de glucosa (P99 de train): {tope:.1f} mg/dL")

# 3) Faltantes: mediana calculada en train y aplicada a train y test
# cigsPerDay: los 29 faltantes son fumadores actuales (S1); la mediana general es 0, así que se usa la de los fumadores de train
fum_train = df.loc[X_train.index, "currentSmoker"] == 1
medianas = X_train.median()
medianas["cigsPerDay"] = X_train.loc[fum_train, "cigsPerDay"].median()
print(f"Mediana de cigarrillos por día (fumadores, train): {medianas['cigsPerDay']:.0f}")
X_train = X_train.fillna(medianas)
X_test = X_test.fillna(medianas)
print("Faltantes restantes (train, test):", int(X_train.isna().sum().sum()), int(X_test.isna().sum().sum()))

# 4) Estandarizar con media y desviación del train
scaler = StandardScaler().fit(X_train)
X_train_sc = pd.DataFrame(scaler.transform(X_train), columns=predictores, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=predictores, index=X_test.index)
X_train_sc.describe().round(2).loc[["mean", "std"]]

Train: 2.966 | Test: 1.272
Prevalencia train: 0.1521 | test: 0.1517
Tope de glucosa (P99 de train): 187.1 mg/dL
Faltantes restantes (train, test): 0 0


,age,sysBP,male,cigsPerDay,glucose
mean,-0.0,0.0,-0.0,-0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0


**Lectura.** La partición deja 2.966 participantes en train y 1.272 en test con la misma prevalencia de CHD (15,21 % y 15,17 %): la estratificación evita que el azar deje pocos casos en alguno de los conjuntos. Dividimos **antes** de imputar y estandarizar para evitar fuga de información: las medianas, el tope y la escala salen solo del train y se aplican al test, que representa datos nuevos. La glucosa se acota al percentil 99 de train (187,1 mg/dL; en todo el dataset es 175, según S2): el valor difiere porque se calcula con el 70 % de los datos y esa cola es muy rala. El tope limita el peso de los casos extremos sin eliminarlos. Se imputa con la mediana porque es robusta a la asimetría de la glucosa y conserva los 4.238 casos; su costo es reducir artificialmente la variabilidad, aceptable en una formativa (S3 comparará con imputación por regresión). En cigarrillos por día se usa la mediana de los **fumadores** de train, porque los 29 faltantes son fumadores actuales y la mediana general es 0, que los habría tratado como no fumadores. Tras estandarizar, los predictores tienen media 0 y desviación 1 en train, de modo que los coeficientes se interpretarán por desviación estándar y serán comparables entre sí.

## 2. Modelo de regresión logística (30 pts)
### 2.1 Selección de variables (3 a 5)
Se seleccionan **cinco predictores** con un criterio doble: que la asociación con CHD aparezca en S1 y, cuando S2 la validó, que resista el remuestreo.

| Variable | r con CHD (S1) | IC bootstrap (S2) | Justificación |
|---|---|---|---|
| Edad | 0,225 | [0,197; 0,254] | Factor de riesgo principal; estable en S2. |
| Presión sistólica | 0,216 | [0,183; 0,250] | Segunda asociación más fuerte; estable en S2. Representa a todo el bloque de presión arterial. |
| Glucosa | 0,126 | no validada | Marcador metabólico; S2 recomendó acotar su cola, lo que se hace en 1.2. |
| Sexo masculino | 0,088 | no validada | Mayor riesgo cardiovascular en hombres; casi no se correlaciona con los demás predictores. |
| Cigarrillos por día | 0,058 | no validada | Captura el tabaquismo: S2 no distingue de cero la correlación marginal, pero eso no permite descartar su aporte en un modelo multivariable. |

**Variables excluidas.** La presión diastólica (r = 0,78 con la sistólica) y la hipertensión prevalente (r = 0,70) forman un bloque colineal con la sistólica (S2), así que entra solo una; diabetes se solapa con la glucosa (r = 0,62, S2); colesterol total e IMC tienen asociaciones débiles (0,082 y 0,075). Con unos 451 casos de CHD en train, hay cerca de 90 por predictor, margen amplio para ajustar cinco variables.

### 2.2 Ajuste e interpretación
TODO (Ninoska): resumen del modelo, coeficientes, odds ratios y significancia.

In [ ]:
# TODO (Ninoska): sm.Logit(...).fit(); odds ratios con IC

### 2.3 Diagnóstico: VIF
TODO (Ninoska): VIF y qué se haría si hubiera multicolinealidad.

In [ ]:
# TODO (Ninoska): VIF

## 3. Evaluación de desempeño predictivo (30 pts)
### 3.1 Matriz de confusión y métricas
TODO (Felipe): umbral 0,5; accuracy, precision, recall y F1 interpretados en el contexto (prevalencia 15,2 %).

In [ ]:
# TODO (Felipe): predicciones en test, matriz de confusión, métricas

### 3.2 Curva ROC y AUC
TODO (Ninoska): curva ROC, AUC e interpretación según los rangos de referencia.

In [ ]:
# TODO (Ninoska): ROC y AUC

## 4. Síntesis y reflexión (15 pts, máx. 1 página)
TODO (Felipe y Ninoska): cómo se integraron S1 y S2, evaluación del desempeño y mejoras para S3.

## Checklist de entrega
- [ ] Ejecuta sin errores de principio a fin y con salidas visibles
- [ ] Hallazgos de S1 y S2 documentados
- [ ] Variables justificadas con S1/S2
- [ ] Modelo, odds ratios y VIF
- [ ] Matriz de confusión, métricas, ROC y AUC
- [ ] Reflexión
- [ ] Semilla fijada (SEED = 42)